In [65]:
import pyodbc
import pygrametl

# Update these with your actual SSMS details
STAGING_CONNECTION_STRING = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=DESKTOP-AR905LJ;"
    "Database=staging;"
    "UID=sa;"
    "PWD=ghom3220;"
)
DWH_CONNECTION_STRING = ( 
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=DESKTOP-AR905LJ;"
    "Database=DWH;"
    "UID=sa;"
    "PWD=ghom3220;"
)

In [66]:
source_conn = pyodbc.connect(STAGING_CONNECTION_STRING)
source_connection = pygrametl.ConnectionWrapper(source_conn)
source_cursor = source_connection.cursor()
source_cursor.execute("SELECT @@version;")
result = source_cursor.fetchone()
print(f"connected to {result[0]}!")

connected to Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)
!


In [67]:
dwh_conn = pyodbc.connect(DWH_CONNECTION_STRING)
dwh_connection = pygrametl.ConnectionWrapper(dwh_conn)
dwh_cursor = dwh_connection.cursor()
dwh_cursor.execute("SELECT @@version;")
result = dwh_cursor.fetchone()
print(f"connected to {result[0]}!")
dwh_connection.setasdefault()

connected to Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)
!


In [68]:
create_table_sql = """
IF NOT EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'[dbo].[Dim_Customer]') AND type in (N'U'))
BEGIN
    CREATE TABLE Dim_Customer (
        Client_ID VARCHAR(50) PRIMARY KEY,
        Full_Name VARCHAR(100),
        Gender VARCHAR(10),
        Age INT,
        Age_Group VARCHAR(20),
        Customer_Type VARCHAR(50),
        Registration_Date DATE
    );
END
"""
dwh_cursor.execute(create_table_sql)
dwh_connection.commit()

In [69]:
from pygrametl.tables import Dimension
Dim_Customer = pygrametl.tables.Dimension(
    name='Dim_Customer',

    key='Client_ID',

    attributes=[
        'Full_Name',
        'Gender',
        'Age',
        'Age_Group',
        'Customer_Type',
        'Registration_Date'
    ]
)    

In [76]:
client_source=source_cursor.execute("SELECT Distinct ClientID,NomClient,Sexe,Age,TrancheAge,ClientType,DateInscription FROM source")
for row in client_source:
    client_data = {
        "Client_ID" : row.ClientID,
        "Full_Name" : row.NomClient,
        "Gender" : row.Sexe,
        "Age" : row.Age,
        "Age_Group" : row.TrancheAge,
        "Customer_Type" : row.ClientType,
        "Registration_Date" : row.DateInscription
    }
    print(client_data)
    Dim_Customer.ensure(client_data)
dwh_connection.commit()
dwh_cursor.close()
dwh_connection.close()

{'Client_ID': 'CLT00001', 'Full_Name': 'Youssef Yahyaoui', 'Gender': 'M', 'Age': 24, 'Age_Group': '18-25', 'Customer_Type': 'VIP', 'Registration_Date': '2022-07-15'}
{'Client_ID': 'CLT00002', 'Full_Name': 'Khadija Jaziri', 'Gender': 'F', 'Age': 33, 'Age_Group': '25-35', 'Customer_Type': 'Régulier', 'Registration_Date': '2020-10-05'}
{'Client_ID': 'CLT00003', 'Full_Name': 'Hatem Mejri', 'Gender': 'M', 'Age': 70, 'Age_Group': '65+', 'Customer_Type': 'Occasionnel', 'Registration_Date': '2020-10-30'}
{'Client_ID': 'CLT00004', 'Full_Name': 'Aicha Feki', 'Gender': 'F', 'Age': 47, 'Age_Group': '45-55', 'Customer_Type': 'VIP', 'Registration_Date': '2020-01-22'}
{'Client_ID': 'CLT00005', 'Full_Name': 'Zied Nciri', 'Gender': 'M', 'Age': 30, 'Age_Group': '25-35', 'Customer_Type': 'Fidèle', 'Registration_Date': '2020-01-26'}
{'Client_ID': 'CLT00006', 'Full_Name': 'Ahmed Ben Salem', 'Gender': 'M', 'Age': 68, 'Age_Group': '65+', 'Customer_Type': 'VIP', 'Registration_Date': '2020-10-25'}
{'Client_ID'